# SmartFit LoRA on DressCode: IDM-VTON (spec §8.4, §23.13)

A seeded DressCode subset, 2,800 pairs each of **tops**, **bottoms** (trousers, skirts) and **dresses**: garment
types IDM-VTON wasn't trained on (its VITON-HD training is tops only; our VITON-HD run gave no held-out gain).
One pass, updates of 16 (525 updates, about 10 h). The same notebook runs every session: it continues from
the previous session's newest checkpoint and, once training is complete, compares base vs LoRA on 48
held-out DressCode test pairs (16 per category).

**Every session** (right-hand panel):
1. **Accelerator:** **GPU T4 x2**. **Internet:** On.
2. **Input → Add Input:** your private *SmartFit LoRA DressCode pack* (code, manifests and images in one).
3. **Session 2 onwards, also:** Input → Add Input → **Your Work** → this notebook → its latest version's output.
4. **Save Version → Save & Run All (Commit)**.

Expected: session 1 trains (~10 h, stops with a checkpoint if the time budget is reached); the evaluation
(~1.5 h) runs in the session after training completes.

In [ ]:
import glob, json, os, shutil, time

SESSION_START = time.time()
hits = [h for h in sorted(glob.glob("/kaggle/input/**/data/dresscode_pack.json", recursive=True))]
assert hits, "DressCode pack not found: add the SmartFit LoRA DressCode pack dataset as an input."
DATA = os.path.dirname(hits[0])
PACK = os.path.dirname(DATA)
PLAN = json.load(open(f"{PACK}/plan.json"))
print("pack:", PACK)
print(json.dumps(PLAN, indent=2))
records = [json.loads(l) for l in open(f"{PACK}/manifests/dresscode_train.jsonl")]
fields = ("person_image_path", "cloth_image_path", "agnostic_mask_path", "densepose_path")
missing = [r[f] for r in records for f in fields if not os.path.exists(os.path.join(DATA, r[f]))]
assert not missing, f"{len(missing)} files missing from the pack, e.g. {missing[:3]}"
print(f"all {len(records)} training samples present")

OUT = "/kaggle/working/lora_dresscode"
os.makedirs(f"{OUT}/checkpoints", exist_ok=True)
step_of = lambda state: int(os.path.basename(os.path.dirname(state)).split("-")[1])
mine = glob.glob(f"{OUT}/checkpoints/step-*/state.pt")
prev = glob.glob("/kaggle/input/**/lora_dresscode/checkpoints/step-*/state.pt", recursive=True)
if prev and (not mine or step_of(max(prev, key=step_of)) > step_of(max(mine, key=step_of))):
    newest = max(prev, key=step_of)
    src = os.path.dirname(newest)
    shutil.copytree(src, f"{OUT}/checkpoints/{os.path.basename(src)}", dirs_exist_ok=True)
    prev_out = os.path.dirname(os.path.dirname(src))
    for name in ("metrics.jsonl", "run.json"):
        if os.path.exists(f"{prev_out}/{name}"):
            shutil.copy(f"{prev_out}/{name}", f"{OUT}/{name}")
    if os.path.isdir(f"{prev_out}/adapter"):
        shutil.copytree(f"{prev_out}/adapter", f"{OUT}/adapter", dirs_exist_ok=True)
    print(f"Continuing from step {step_of(newest)} of {PLAN['max_steps']} ({src})")
elif mine:
    print(f"Continuing from step {step_of(max(mine, key=step_of))} (this session's own checkpoint)")
else:
    print(f"Starting a new run: {PLAN['max_steps']} updates")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code at a fixed commit and an isolated environment

In [ ]:
![ -d /tmp/IDM-VTON ] || git clone -q https://github.com/yisol/IDM-VTON.git /tmp/IDM-VTON
!git -C /tmp/IDM-VTON checkout -q 0d5f3ec2d737487a9bb24e4100936ad254780383 && git -C /tmp/IDM-VTON log -1 --format="%H %cd"
!pip install -q uv==0.5.11
![ -x /tmp/env/bin/python ] || uv venv -q --python 3.11 /tmp/env
!uv pip install -q --python /tmp/env/bin/python torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install -q --python /tmp/env/bin/python -r {PACK}/code/requirements-lora.txt
!/tmp/env/bin/python -c "import torch, peft; print(torch.__version__, 'peft', peft.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"

## 2. Train (continues automatically; stops at the session's time budget with a checkpoint)

In [ ]:
done_before = os.path.exists(f"{OUT}/run.json") and json.load(open(f"{OUT}/run.json")).get("status") == "completed"
if done_before and os.path.isdir(f"{OUT}/adapter"):
    print("Training already completed in an earlier session: going straight to evaluation.")
else:
    TRAIN_ARGS = " ".join([
        f"--repo /tmp/IDM-VTON --manifest {PACK}/manifests/dresscode_train.jsonl --data-root {DATA} --out {OUT}",
        "--resume auto", f"--max-minutes {PLAN['session_train_minutes']}",
        f"--max-steps {PLAN['max_steps']} --grad-accum {PLAN['grad_accum']} --warmup-steps {PLAN['warmup_steps']}",
        f"--save-every {PLAN['save_every']} --log-every {PLAN['log_every']} --augment {PLAN['augment']} --num-workers 2",
    ])
    !/tmp/env/bin/python {PACK}/code/train_lora.py {TRAIN_ARGS}

In [ ]:
import matplotlib.pyplot as plt

run = json.load(open(f"{OUT}/run.json"))
rows = [json.loads(l) for l in open(f"{OUT}/metrics.jsonl")] if os.path.exists(f"{OUT}/metrics.jsonl") else []
FINISHED = run.get("status") == "completed"
print(json.dumps({k: run.get(k) for k in ("status", "final_step", "minutes", "peak_reserved_gb_by_gpu")}, indent=2))
print(f"progress: {run.get('final_step')} / {PLAN['max_steps']} updates")
if rows:
    plt.figure(figsize=(8, 3))
    plt.plot([r["step"] for r in rows], [r["loss"] for r in rows])
    plt.xlabel("update"); plt.ylabel("loss (mean of 5 updates)"); plt.title("LoRA on DressCode"); plt.grid(alpha=.3)
    plt.show()
if not FINISHED:
    print("Not finished yet: run another session with this version's output attached as input.")

## 3. Held-out evaluation (only when training is complete)
48 DressCode test pairs (16 tops, 16 bottoms, 16 dresses), base vs LoRA, same seed and inputs.

In [ ]:
hours_left = 12 - (time.time() - SESSION_START) / 3600
if not FINISHED:
    print("Skipped: training is not complete yet.")
elif hours_left < 1.75:
    print(f"Skipped: only {hours_left:.1f} h left. Run once more with this output attached: training is done, "
          "so the next session goes straight to evaluation.")
else:
    !/tmp/env/bin/python {PACK}/code/eval_lora.py --repo /tmp/IDM-VTON --manifest {PACK}/manifests/heldout_eval.jsonl --data-root {DATA} --adapter {OUT}/adapter --out {OUT}/eval

In [ ]:
import statistics
from IPython.display import Image as Show, display

if os.path.exists(f"{OUT}/eval/results.json"):
    ev = json.load(open(f"{OUT}/eval/results.json"))
    print(json.dumps(ev["summary"], indent=2))
    for cat in ("upper_body", "lower_body", "dresses"):
        rows = [r for r in ev["samples"] if r["sample_id"].split(":")[1] == cat]
        if rows:
            base = statistics.mean(r["base_lpips"] for r in rows); lora = statistics.mean(r["lora_lpips"] for r in rows)
            better = sum(r["lora_lpips"] < r["base_lpips"] for r in rows)
            print(f"{cat:11s} LPIPS base {base:.4f} -> LoRA {lora:.4f}  (LoRA better on {better}/{len(rows)})")
    display(Show(filename=f"{OUT}/eval/sheet.jpg", width=900))

## 4. Download
`results_lora_dresscode.zip` (adapter, metrics, run summary, evaluation; no checkpoints). When the run is finished and evaluated, download it into the repo's `backend/` folder.

In [ ]:
!cd /kaggle/working && rm -f results_lora_dresscode.zip && zip -qr results_lora_dresscode.zip lora_dresscode -x "lora_dresscode/checkpoints/*" && ls -la /kaggle/working /kaggle/working/lora_dresscode